In [1]:
import pandas as pd
import numpy as np
import json

In [2]:
with open("ESV DATA.txt", "r") as file:
    raw_text = file.read()
data = json.loads(raw_text)
all_records = []
for stock_symbol, stock_data in data.items():
    for entry in stock_data.get("DATA", []):
        flat_entry = {}
        flat_entry['STOCK'] = stock_symbol
        flat_entry['DateTime'] = entry.get('DateTime', {}).get('$date', None)
        for timeframe in ['5m', '15m', 'D', 'H', 'M', 'W']:
            if timeframe in entry:
                for k, v in entry[timeframe].items():
                    flat_entry[f"{timeframe}.{k}"] = v

        all_records.append(flat_entry)
df_ESV = pd.DataFrame(all_records)
df_ESV['DateTime'] = pd.to_datetime(df_ESV['DateTime'], errors='coerce')
df_ESV['DateTime'] = pd.to_datetime(df_ESV['DateTime']).dt.date

In [3]:
with open("HOVER DATA.txt", "r") as file:
    raw_text = file.read()
data = json.loads(raw_text)

# Manual flattening
all_records = []
for stock_symbol, stock_data in data.items():
    for entry in stock_data.get("DATA", []):
        flat_entry = {}

        # Basic info
        flat_entry['STOCK'] = stock_symbol

        # DateTime extraction
        flat_entry['DateTime'] = entry.get('DateTime', {}).get('$date', None)

        # Flatten each timeframe: 5m, 15m, D, H, M, W
        for timeframe in ['5m', '15m', 'D', 'H', 'M', 'W']:
            if timeframe in entry:
                for k, v in entry[timeframe].items():
                    flat_entry[f"{timeframe}.{k}"] = v

        all_records.append(flat_entry)

# Final DataFrame
df_HOVER = pd.DataFrame(all_records)

# Parse DateTime
df_HOVER['DateTime'] = pd.to_datetime(df_HOVER['DateTime'], errors='coerce')
df_HOVER['DateTime'] = pd.to_datetime(df_HOVER['DateTime']).dt.date

In [4]:
df_HOVER

,STOCK,DateTime,5m.Prev_5m_PerChange,5m.Prev_Prev_5m_PerChange,5m.Prev_5m_close,5m.perc_chg,5m.InOutEdge,5m.MP20,5m.MP50,5m.MP200,...,W.BLPVT,W.BELPVT,W.BULCO,W.BELCO,W.per_volume_change,W.open_equals_high,W.open_equals_low,W.M_RSI,W.M_EMA20,W.M_EMA50
0,BSE,2023-02-08,0.04,0.21,487.5,-0.16,,5mPB20,5mPB50,5mPB200,...,,,,BELCO,-41.01,False,False,,NaN,NaN
1,BSE,2023-02-09,0.05,-0.22,504.85,0.17,I5m,5mPA20,5mPA50,5mPA200,...,,,,BELCO,-1.87,False,False,,NaN,NaN
2,BSE,2023-02-10,-0.46,0.0,499.2,0.36,,5mPA20,5mPB50,5mPA200,...,,,,,23.11,False,False,,NaN,NaN
3,BSE,2023-02-13,-0.16,-0.11,488.25,0.16,I5m,5mPB20,5mPB50,5mPB200,...,,,,,-86.66,False,False,,NaN,NaN
4,BSE,2023-02-14,-0.03,-0.1,481.3,0.04,,5mPB20,5mPB50,5mPB200,...,,,,,-73.34,False,False,,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64293,360ONE,2023-08-24,0.02,-0.08,512.1,0.0,,5mPB20,5mPB50,5mPA200,...,,,,,31.12,False,False,,NaN,NaN
64294,360ONE,2023-08-25,0.08,0.14,510.05,-0.08,I5m,5mPA20,5mPA50,5mPB200,...,,,,,77.64,False,False,,NaN,NaN
64295,360ONE,2023-08-28,-0.19,0.04,505.25,0.07,,5mPB20,5mPB50,5mPB200,...,,,,,-84.82,False,False,,NaN,NaN
64296,360ONE,2023-08-29,-0.43,0.0,506.25,0.16,,5mPA20,5mPA50,5mPA200,...,,,,,-74.29,True,False,,NaN,NaN


In [8]:
hover_stocks = set(df_HOVER['STOCK'].unique())
esv_stocks = set(df_ESV['STOCK'].unique())
common_stocks = hover_stocks.intersection(esv_stocks)
hover_only = hover_stocks - esv_stocks
esv_only = esv_stocks - hover_stocks

In [9]:
len(common_stocks), len(hover_only), len(esv_only)

(543, 0, 0)

In [14]:
print(df_HOVER.columns.tolist())

['STOCK', 'DateTime', '5m.Prev_5m_PerChange', '5m.Prev_Prev_5m_PerChange', '5m.Prev_5m_close', '5m.perc_chg', '5m.InOutEdge', '5m.MP20', '5m.MP50', '5m.MP200', '5m.MCo', '5m.MCO', '5m.RSI C', '5m.RSI', '5m.RSI KC', '5m.RSI DC', '5m.RSI KD', '5m.RSI KCD', '5m.MACD', '5m.BLPVT', '5m.BELPVT', '5m.BULCO', '5m.BELCO', '5m.per_volume_change', '5m.open_equals_high', '5m.open_equals_low', '15m.Prev_15m_PerChange', '15m.Prev_Prev_15m_PerChange', '15m.Prev_15m_close', '15m.perc_chg', '15m.InOutEdge', '15m.MP20', '15m.MP50', '15m.MP200', '15m.MCo', '15m.MCO', '15m.RSI C', '15m.RSI', '15m.RSI KC', '15m.RSI DC', '15m.RSI KD', '15m.RSI KCD', '15m.MACD', '15m.BLPVT', '15m.BELPVT', '15m.BULCO', '15m.BELCO', '15m.per_volume_change', '15m.open_equals_high', '15m.open_equals_low', 'D.perc_chg', 'D.InOutEdge', 'D.MP20', 'D.MP50', 'D.MP200', 'D.MCo', 'D.MCO', 'D.RSI C', 'D.RSI', 'D.RSI KC', 'D.RSI DC', 'D.RSI KD', 'D.RSI KCD', 'D.MACD', 'D.BLPVT', 'D.BELPVT', 'D.BULCO', 'D.BELCO', 'D.per_volume_change', 'D

In [11]:
print(df_ESV.columns.tolist())

['STOCK', 'DateTime', '5m.perc_chg', '5m.per_volume_change', '5m.PDEMA20', '5m.PDEMA50', '5m.PDSMA200', '5m.PDVWAP', '5m.EMA20D50', '5m.EMA20D200', '5m.EMA50D200', '5m.20DVWAP', '5m.50DVWAP', '5m.200DVWAP', '5m.RISING_RSI', '5m.RISING_LTP', '5m.RISING_VWAP', '5m.DECLINING_RSI', '5m.DECLINING_LTP', '5m.DECLINING_VWAP', '5m.RISING_VOLUME', '5m.DECLINING_VOLUME', '5m.RISING_OI', '5m.DECLINING_OI', '5m.LtpChg', '5m.VolChg', '5m.PGL', '5m.N50_PGL', '5m.NFO_PGL', '5m.VOL_PGL', '5m.N50_VOL_PGL', '5m.NFO_VOL_PGL', '5m.OiChg', '5m.OI_PGL', '15m.perc_chg', '15m.per_volume_change', '15m.PDEMA20', '15m.PDEMA50', '15m.PDSMA200', '15m.EMA20D50', '15m.EMA20D200', '15m.EMA50D200', '15m.RISING_RSI', '15m.RISING_LTP', '15m.DECLINING_RSI', '15m.DECLINING_LTP', '15m.RISING_VOLUME', '15m.DECLINING_VOLUME', '15m.RISING_OI', '15m.DECLINING_OI', '15m.LtpChg', '15m.VolChg', '15m.PGL', '15m.N50_PGL', '15m.NFO_PGL', '15m.VOL_PGL', '15m.N50_VOL_PGL', '15m.NFO_VOL_PGL', '15m.OiChg', '15m.OI_PGL', 'D.perc_chg', 'D.

In [5]:
# Ensure DateTime is in datetime format and truncate to just the date
df_HOVER['DateTime'] = pd.to_datetime(df_HOVER['DateTime']).dt.date
df_ESV['DateTime'] = pd.to_datetime(df_ESV['DateTime']).dt.date

# Inner merge on 'STOCK' and 'DateTime'
merged_df = pd.merge(df_HOVER, df_ESV, on=['STOCK', 'DateTime'], how='inner')

In [6]:
# Assuming df is your DataFrame
# columns_to_drop = [col for col in merged_df.columns if col.startswith("D.") or col.startswith("W.")]
#df = merged_df.drop(columns=columns_to_drop)
df = merged_df.replace(r'^\s*$', np.nan, regex=True)

C:\Users\choud\AppData\Local\Temp\ipykernel_13440\1618190993.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = merged_df.replace(r'^\s*$', np.nan, regex=True)


In [ ]:
df = df.dropna(axis=1, how='any')

In [8]:
# 1. Drop short timeframes (5m, 15m, H)
df = df[[col for col in df.columns if not col.startswith(('5m.', '15m.'))]]

# 2. Sort by DateTime and STOCK
df = df.sort_values(by=['DateTime', 'STOCK']).reset_index(drop=True)

# 3. (Optional) Rename DateTime to Date for clarity
df = df.rename(columns={'DateTime': 'Date'})


In [9]:
df.to_csv("merged_data.csv", index=False)

In [80]:
daily_mp200_value = 'DPA200'        
daily_mp50_value = 'DPA50'          
daily_rsi_min = 30                 
daily_rsi_max = 70                 
monthly_rsi_threshold = 50         
weekly_rsi_threshold = 50          
selected_date = '2023-02-08'       
top_n = 5                          

,STOCK,Date,D.perc_chg_x,D.MP20,D.MP50,D.MP200,D.MCo,D.RSI,D.MACD,D.per_volume_change_x,...,W.PDSMA200,W.EMA20D50,W.EMA20D200,W.EMA50D200,W.RISING_RSI,W.RISING_LTP,W.DECLINING_RSI,W.DECLINING_LTP,W.RISING_VOLUME,W.DECLINING_VOLUME


In [103]:


# Apply filtering conditions
filtered_df = df[
    (df['D.MP200'] == daily_mp200_value) &
    (df['D.MP50'] == daily_mp50_value) &
    (df['D.RSI'].between(daily_rsi_min, daily_rsi_max)) &
    (df['M.RSI'] >= monthly_rsi_threshold) &
    (df['W.RSI'] >= weekly_rsi_threshold)
]

In [104]:
sorted_df = filtered_df.sort_values(
    by=['D.PDEMA20', 'D.RSI'],
    ascending=[False, False]   # both columns descending
)

In [105]:
# Filter for the specific date
date_filtered_df = sorted_df[sorted_df['Date'] == selected_date]

# Select the top N rows from the filtered, sorted DataFrame
top_stocks = date_filtered_df.head(top_n)


In [106]:
# Define which columns to display in the output
display_columns = [
    'STOCK', 'Date',
    'D.MP20', 'D.MP50', 'D.MP200', 'D.RSI',
    'M.RSI', 'W.RSI', 'D.PDEMA20'
]
print(top_stocks[display_columns])


Empty DataFrame
Columns: [STOCK, Date, D.MP20, D.MP50, D.MP200, D.RSI, M.RSI, W.RSI, D.PDEMA20]
Index: []
